# Yolo

In [ ]:
!pip install -q kagglehub

In [ ]:
import os
import kagglehub
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!mkdir -p ~/.kaggle

with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{os.environ["KAGGLE_USERNAME"]}","key":"{os.environ["KAGGLE_KEY"]}"}}')

!chmod 600 ~/.kaggle/kaggle.json

dl_lab_2_stuff_detection_path = kagglehub.competition_download('dl-lab-2-stuff-detection')

print(f'SUCCESS: Data source imported to {dl_lab_2_stuff_detection_path}')

SUCCESS: Data source imported to /root/.cache/kagglehub/competitions/dl-lab-2-stuff-detection


In [ ]:
!pip install -q ultralytics

In [ ]:
import os
from ultralytics import YOLO

model = YOLO("yolo26x.pt")

model_path = "yolo26x.pt"
model_size = os.path.getsize(model_path) / (1024 * 1024)

print(f'STATUS: YOLO26x initialized')
print(f'Path: {os.path.abspath(model_path)}')
print(f'Size: {model_size:.2f} MB')

STATUS: YOLO26n initialized
Path: /content/yolo26n.pt
Size: 5.29 MB


In [ ]:
!pip install -q psutil

In [ ]:
import yaml
import torch
import shutil
import random
from ultralytics import YOLO

base_path = '/root/.cache/kagglehub/competitions/dl-lab-2-stuff-detection'
source_images = f'{base_path}/yolo_dataset/yolo_dataset/train/images'
source_labels = f'{base_path}/yolo_dataset/yolo_dataset/train/labels'

train_dir = '/content/data_split/train'
val_dir = '/content/data_split/val'

os.makedirs(f'{train_dir}/images', exist_ok=True)
os.makedirs(f'{train_dir}/labels', exist_ok=True)
os.makedirs(f'{val_dir}/images', exist_ok=True)
os.makedirs(f'{val_dir}/labels', exist_ok=True)

image_files = [f for f in os.listdir(source_images) if f.endswith(('.jpg', '.png', '.jpeg'))]
random.seed(42)
random.shuffle(image_files)

split_idx = int(len(image_files) * 0.8)
train_files = image_files[:split_idx]
val_files = image_files[split_idx:]

for f in train_files:
    shutil.copy(f'{source_images}/{f}', f'{train_dir}/images/{f}')
    label_file = f.replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt')
    if os.path.exists(f'{source_labels}/{label_file}'):
        shutil.copy(f'{source_labels}/{label_file}', f'{train_dir}/labels/{label_file}')

for f in val_files:
    shutil.copy(f'{source_images}/{f}', f'{val_dir}/images/{f}')
    label_file = f.replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt')
    if os.path.exists(f'{source_labels}/{label_file}'):
        shutil.copy(f'{source_labels}/{label_file}', f'{val_dir}/labels/{label_file}')

data_config = {
    'path': '/content/data_split',
    'train': 'train/images',
    'val': 'val/images',
    'nc': 2,
    'names': ['customer', 'staff']
}

new_data_path = '/content/data.yaml'
with open(new_data_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f'Train: {len(train_files)} images')
print(f'Val: {len(val_files)} images')

model = YOLO("yolo26x.pt")

results = model.train(
    data=new_data_path,
    epochs=50,
    batch=8,
    imgsz=640,
    workers=2,
    device=0,
    cache=True,
    optimizer='AdamW',
    lr0=0.001,
    patience=20,
    seed=42
)

print(f'Model saved to: {results.save_dir}')
print(f'Best model: {results.save_dir}/weights/best.pt')

Train: 3126 images
Val: 782 images
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pat

In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path

base_path = '/root/.cache/kagglehub/competitions/dl-lab-2-stuff-detection'
model_path = '/content/runs/detect/miet/lab2_split_yolo26n/weights/best.pt'

print('File check:')
test_images_path = f'{base_path}/test_images/test_images'
if not os.path.exists(test_images_path):
    print(f'ERROR: Test images not found: {test_images_path}')
    sys.exit(1)
else:
    test_images = [f for f in os.listdir(test_images_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    print(f'Test images: {len(test_images)} files')

if not os.path.exists(model_path):
    print(f'ERROR: Model not found: {model_path}')
    print('Train model first')
    sys.exit(1)
else:
    print(f'Model: {model_path}')

sample_files = [f for f in os.listdir(base_path) if 'sample' in f.lower() and f.endswith('.csv')]
if not sample_files:
    print(f'ERROR: No sample submission CSV in {base_path}')
    sys.exit(1)
else:
    sample_file = sample_files[0]
    print(f'Sample submission: {sample_file}')

    df = pd.read_csv(f'{base_path}/{sample_file}')
    if 'image_name' not in df.columns:
        print(f'ERROR: Column "image_name" missing')
        print(f'Columns: {df.columns.tolist()}')
        sys.exit(1)
    print(f'Rows: {len(df)}')

def build_submission_from_solution_order(
    solution_csv: str,
    preds_dir: str,
    output_csv: str = "submission.csv",
    image_col: str = "image_name",
    boxes_col: str = "boxes",
    row_id_col: str = "id",
    require_score: bool = True,
    clamp_score: bool = True,
    keep_only_class: int | None = None,
) -> None:
    sol_path = Path(solution_csv)
    pdir = Path(preds_dir)

    if not sol_path.exists():
        raise FileNotFoundError(f"solution_csv not found: {sol_path}")
    if not pdir.exists() or not pdir.is_dir():
        raise FileNotFoundError(f"preds_dir not found or not a dir: {pdir}")

    sol = pd.read_csv(sol_path)
    if image_col not in sol.columns:
        raise ValueError(f"solution.csv must contain column '{image_col}'")

    image_names = sol[image_col].astype(str).tolist()

    rows = []
    missing_files = []
    for idx, image_name in enumerate(image_names):
        stem = Path(image_name).stem
        pred_file = pdir / f"{stem}.txt"

        boxes = []
        if pred_file.exists():
            content = pred_file.read_text(encoding="utf-8", errors="replace").strip()
            if content:
                for ln in content.splitlines():
                    ln = ln.strip()
                    if not ln:
                        continue
                    parts = ln.split()
                    if require_score and len(parts) < 6:
                        continue
                    if len(parts) < 5:
                        continue

                    try:
                        cls = int(float(parts[0]))
                        if keep_only_class is not None and cls != keep_only_class:
                            continue

                        xc = float(parts[1])
                        yc = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])
                        sc = float(parts[5]) if len(parts) >= 6 else 1.0
                    except ValueError:
                        continue

                    if clamp_score:
                        sc = 0.0 if sc < 0.0 else (1.0 if sc > 1.0 else sc)

                    boxes.append([xc, yc, w, h, sc])
        else:
            missing_files.append(image_name)

        rows.append(
            {
                row_id_col: idx,
                image_col: image_name,
                boxes_col: json.dumps(boxes, separators=(",", ":")),
            }
        )

    if missing_files:
        print(f'WARNING: {len(missing_files)} prediction files missing')

    sub = pd.DataFrame(rows, columns=[row_id_col, image_col, boxes_col])
    sub.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv} ({len(sub)} rows)")


In [ ]:
model = YOLO(model_path)

print('\nRunning inference...')
results = model.predict(f'{base_path}/test_images/test_images', save_txt=True, save_conf=True, stream=True)
for r in results:
    pass

print('\nBuilding submission...')
build_submission_from_solution_order(
    solution_csv=f'{base_path}/{sample_file}',
    preds_dir='/content/runs/detect/predict/labels',
    output_csv='/content/submission.csv',
    keep_only_class=1
)